# Exp 06: Diagnostics

目标：用一个故意写坏的训练循环，串起 `torch.compile` 诊断工具链。

诊断顺序：
1. `counters["stats"]["unique_graphs"]`：先确认是否有 recompile 异常。
2. `TORCH_LOGS=recompiles`：定位哪个 frame、哪个 guard 失败。
3. `torch._dynamo.explain`：统计 graph break 数量和原因。
4. `TORCH_LOGS=dynamic` / `recompiles_verbose`：深入看 symbolic shape 和所有 cache entry。
5. 逐个修复，再验证 `unique_graphs` 是否下降。

脚本版运行示例：

```bash
python experiments_py/exp06_diagnostics.py --phase all
TORCH_LOGS=recompiles python experiments_py/exp06_diagnostics.py --phase broken
```

In [ ]:
import random
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch._dynamo.utils import counters

assert torch.cuda.is_available(), "This notebook requires a CUDA GPU."

DEVICE = "cuda"
DTYPE = torch.bfloat16
IN_DIM = 256
OUT_DIM = 64
STEPS = 25

torch.manual_seed(42)
random.seed(42)

## 坏版本：四个问题同时出现

这个模型故意包含四个 bad smell：

- P1：`dynamic=False` 但 batch size 每步随机变化。
- P2：每步重新赋值 `self.qk_mask`，导致 object id 变化，触发 `ID_MATCH` guard 失败。
- P3：`scale` 是 Python `int` 参数，不同值触发 `EQUALS_MATCH` guard。
- P4：forward 里用 `.item()` 和 `print` 做 data-dependent 副作用，触发 graph break。

In [ ]:
class BrokenModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.fc = nn.Linear(IN_DIM, OUT_DIM)
        self.qk_mask = torch.ones(1, device=DEVICE, dtype=DTYPE)

    def forward(self, x: torch.Tensor, scale: int = 1) -> torch.Tensor:
        h = self.fc(x) * scale

        if h.abs().max().item() > 10.0:
            print("large activation detected")
            h = h.clamp(-10.0, 10.0)

        h = h * self.qk_mask
        return h.mean(dim=-1)


def clear_grads(model: nn.Module) -> None:
    for param in model.parameters():
        param.grad = None


def run_broken(steps: int = STEPS) -> tuple[int, float]:
    model = BrokenModel().to(DEVICE, dtype=DTYPE)
    compiled = torch.compile(model, mode="default", fullgraph=False, dynamic=False)
    scales = [1, 2, 3, 1, 2, 3]

    counters.clear()
    start = time.perf_counter()

    for step in range(steps):
        batch_size = random.choice([8, 16, 24, 32])
        x = torch.randn(batch_size, IN_DIM, device=DEVICE, dtype=DTYPE)
        y = torch.randn(batch_size, OUT_DIM, device=DEVICE, dtype=DTYPE)
        scale = scales[step % len(scales)]

        model.qk_mask = torch.ones(1, device=DEVICE, dtype=DTYPE) * (step % 3 + 1)

        loss = F.mse_loss(compiled(x, scale), y)
        loss.backward()
        clear_grads(model)

    torch.cuda.synchronize()
    elapsed_ms = (time.perf_counter() - start) * 1000
    return counters["stats"]["unique_graphs"], elapsed_ms

In [ ]:
broken_graphs, broken_ms = run_broken()
print(f"broken unique_graphs={broken_graphs}  total_time={broken_ms:.0f} ms")

if broken_graphs > 5:
    print("明显存在 recompile 问题。下一步建议打开 TORCH_LOGS=recompiles 查看失败 guard。")

## `torch._dynamo.explain` 看 graph break

`explain` 会对函数做一次 Dynamo 分析，返回 graph 数量、break 数量、捕获 op 数，以及 break 原因。

它不替代 `TORCH_LOGS=recompiles`，因为它主要看 graph break，不完整展示所有 guard failure；但它很适合快速定位 `.item()`、`print`、不支持的 Python 行为。

In [ ]:
model_for_explain = BrokenModel().to(DEVICE, dtype=DTYPE)
x_explain = torch.randn(16, IN_DIM, device=DEVICE, dtype=DTYPE)
explanation = torch._dynamo.explain(model_for_explain)(x_explain, 2)

print(f"graph_count       = {explanation.graph_count}")
print(f"graph_break_count = {explanation.graph_break_count}")
print(f"op_count          = {explanation.op_count}")
print("break_reasons:")
for idx, reason in enumerate(explanation.break_reasons):
    print(f"[{idx}] {reason}")

## TORCH_LOGS 读法速查

常见 guard failure 和对应修复：

- `tensor 'x' size mismatch at index 0`：batch 维变化。修复：`mark_dynamic(x, 0, min=2, max=...)` 或 bucketing。
- `___check_obj_id(self.qk_mask, ...)`：对象 id 变化。修复：`register_buffer` 预分配，并用 in-place 更新。
- `L['scale'] == 2`：Python scalar 值变化。修复：改成 0-dim tensor。
- `.item() call was found`：data-dependent Python 分支。修复：`torch.cond`、`is_compiling()` 或隔离外层 eager。

## 修复版本

修复策略逐一对应：

- P1：`dynamic=None` + `mark_dynamic`，让 batch 维从第一次调用就是 dynamic。
- P2：`qk_mask` 改为 buffer，并使用 `fill_` 原地更新，保持 object id 不变。
- P3：`scale` 改成 0-dim tensor。
- P4：`print` 用 `is_compiling()` 保护，`clamp` 改为无条件 tensor 运算。

In [ ]:
class FixedModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.fc = nn.Linear(IN_DIM, OUT_DIM)
        self.register_buffer("qk_mask", torch.ones(1, dtype=DTYPE))

    def forward(self, x: torch.Tensor, scale: torch.Tensor) -> torch.Tensor:
        h = self.fc(x) * scale

        if not torch.compiler.is_compiling():
            if h.abs().max().item() > 10.0:
                print("large activation detected in eager mode")

        h = h.clamp(-10.0, 10.0)
        h = h * self.qk_mask
        return h.mean(dim=-1)


def run_fixed(steps: int = STEPS) -> tuple[int, float]:
    model = FixedModel().to(DEVICE, dtype=DTYPE)
    compiled = torch.compile(model, mode="default", fullgraph=False, dynamic=None)
    scales = [1, 2, 3, 1, 2, 3]

    counters.clear()
    start = time.perf_counter()

    for step in range(steps):
        batch_size = random.choice([8, 16, 24, 32])
        x = torch.randn(batch_size, IN_DIM, device=DEVICE, dtype=DTYPE)
        y = torch.randn(batch_size, OUT_DIM, device=DEVICE, dtype=DTYPE)
        scale = torch.tensor(scales[step % len(scales)], device=DEVICE, dtype=DTYPE)

        torch._dynamo.mark_dynamic(x, 0, min=2, max=128)
        torch._dynamo.mark_dynamic(y, 0, min=2, max=128)
        model.qk_mask.fill_(float(step % 3 + 1))

        loss = F.mse_loss(compiled(x, scale), y)
        loss.backward()
        clear_grads(model)

    torch.cuda.synchronize()
    elapsed_ms = (time.perf_counter() - start) * 1000
    return counters["stats"]["unique_graphs"], elapsed_ms

In [ ]:
fixed_graphs, fixed_ms = run_fixed()
print(f"broken: unique_graphs={broken_graphs:3d}, total_time={broken_ms:.0f} ms")
print(f"fixed : unique_graphs={fixed_graphs:3d}, total_time={fixed_ms:.0f} ms")

if fixed_ms > 0:
    print(f"time ratio broken/fixed = {broken_ms / fixed_ms:.2f}x")

## 小结

诊断时不要先猜优化开关，先确认事实：

1. 看 `unique_graphs` 是否异常增长。
2. 用 `TORCH_LOGS=recompiles` 找失败 guard。
3. 用 `torch._dynamo.explain` 找 graph break。
4. 对 shape、对象 id、Python scalar、data-dependent 分支分别修复。
5. 修复后再次观察 `unique_graphs`，确认问题真的消失。

更重的离线分析可以使用 `TORCH_TRACE=/tmp/trace python ...` 生成 trace，再用 `tlparse` 查看完整编译时间线。